## 🛠️ Step 1: Programmatic Google Drive Workspace Organization
We will mount Google Drive and ensure the directory structure exists for our simulation inputs and outputs.

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define paths
base_path = '/content/drive/MyDrive/fifa-wc2026-predictor/data_pipeline/04_tournament_simulation'
input_path = os.path.join(base_path, 'input')
output_path = os.path.join(base_path, 'output')

# Create directories
for path in [input_path, output_path]:
    os.makedirs(path, exist_ok=True)

print(f"Workspace organized at: {base_path}")

Mounted at /content/drive
Workspace organized at: /content/drive/MyDrive/fifa-wc2026-predictor/data_pipeline/04_tournament_simulation


## 🛠️ Step 2: The Qualified 48 Filter
We load the calibrated strengths and the official squad lists to filter our simulation entities.

In [ ]:
import pandas as pd
import os
from IPython.display import display, Markdown

strengths_path = '/content/drive/MyDrive/fifa-wc2026-predictor/data_pipeline/03_model_training/output/simulation_team_strengths.csv'
squad_list_path = '/content/drive/MyDrive/FIFA_Predictor_Data/SquadLists.csv'

if not os.path.exists(strengths_path): raise FileNotFoundError(strengths_path)
if not os.path.exists(squad_list_path): raise FileNotFoundError(squad_list_path)

df_strengths = pd.read_csv(strengths_path)
df_qualified = pd.read_csv(squad_list_path)

# STEP 1: Implement the Hardcoded Dictionary (CORRECTED)
entity_mapping = {
    "Bosnia and Herzegovina": "Bosnia And Herzegovina",
    "Cape Verde": "Cabo Verde",
    "DR Congo": "Congo DR",
    "Czech Republic": "Czechia",
    "Ivory Coast": "Côte D'Ivoire",
    "Iran": "IR Iran",
    "South Korea": "Korea Republic",
    "Turkey": "Türkiye",
    "United States": "USA",
    "Curacao": "Curaçao"
}

# Normalize and apply mapping
df_strengths['team_name'] = df_strengths['team_name'].str.strip()
df_strengths['team_name'] = df_strengths['team_name'].replace(entity_mapping)

# STEP 2: The Inner Join and Validation Gate
final_teams = pd.merge(
    df_strengths,
    df_qualified[['Team']].drop_duplicates(),
    left_on='team_name',
    right_on='Team',
    how='inner'
).drop(columns=['Team']).rename(columns={'avg_expected_win_probability': 'strength'})

# Check for missing teams if gate is about to fail
if len(final_teams) != 48:
    qualified_set = set(df_qualified['Team'].unique())
    matched_set = set(final_teams['team_name'].unique())
    print(f"CRITICAL FAILURE: Missing teams: {qualified_set - matched_set}")

assert len(final_teams) == 48, f"CRITICAL LEAK: Expected exactly 48 teams, but got {len(final_teams)}. Check for dropped rows."

print("Validation Passed: 48/48 teams aligned.")
display(final_teams.sort_values(by='strength', ascending=False).head(10))

Validation Passed: 48/48 teams aligned.


,team_name,strength
0,Belgium,0.542542
1,Spain,0.538619
2,Brazil,0.538163
3,France,0.525489
4,Japan,0.521561
5,Côte D'Ivoire,0.520459
6,Portugal,0.513670
7,Germany,0.513020
8,England,0.512858
9,USA,0.511148


In [ ]:
print(df_qualified['Team'].unique())

['Algeria' 'Argentina' 'Australia' 'Austria' 'Belgium'
 'Bosnia And Herzegovina' 'Brazil' 'Cabo Verde' 'Canada' 'Colombia'
 'Congo DR' "Côte D'Ivoire" 'Croatia' 'Curaçao' 'Czechia' 'Ecuador'
 'Egypt' 'England' 'France' 'Germany' 'Ghana' 'Haiti' 'IR Iran' 'Iraq'
 'Japan' 'Jordan' 'Korea Republic' 'Mexico' 'Morocco' 'Netherlands'
 'New Zealand' 'Norway' 'Panama' 'Paraguay' 'Portugal' 'Qatar'
 'Saudi Arabia' 'Scotland' 'Senegal' 'South Africa' 'Spain' 'Sweden'
 'Switzerland' 'Tunisia' 'Türkiye' 'Uruguay' 'USA' 'Uzbekistan']


## 🛠️ Step 3 & 4: Tournament Engine (Group & Knockout Logic)
We define the functions to handle the probabilistic match outcomes and the bracket progression.

In [ ]:
import numpy as np

def simulate_match(team_a, team_b, strengths_dict):
    s_a = strengths_dict[team_a]
    s_b = strengths_dict[team_b]
    prob_a_wins = s_a / (s_a + s_b)
    return team_a if np.random.random() < prob_a_wins else team_b

def run_tournament(teams_df):
    strengths = dict(zip(teams_df['team_name'], teams_df['strength']))
    all_teams = teams_df['team_name'].tolist()
    np.random.shuffle(all_teams)

    # STEP 4: Standard 2026 Rules (12 Groups of 4)
    groups = [all_teams[i:i+4] for i in range(0, 48, 4)]

    qualified_to_knockout = []
    third_places = []

    for group in groups:
        # Group ranking simulation
        group_sorted = sorted(group, key=lambda x: strengths[x] * np.random.uniform(0.9, 1.1), reverse=True)
        qualified_to_knockout.extend(group_sorted[:2]) # Top 2
        third_places.append(group_sorted[2]) # Collect 3rd place

    # Best 8 third-place teams advance to fill Round of 32
    best_third = sorted(third_places, key=lambda x: strengths[x] * np.random.uniform(0.9, 1.1), reverse=True)
    qualified_to_knockout.extend(best_third[:8])

    # Knockout Phase: Round of 32 down to 1
    bracket = qualified_to_knockout
    while len(bracket) > 1:
        next_round = []
        for i in range(0, len(bracket), 2):
            winner = simulate_match(bracket[i], bracket[i+1], strengths)
            next_round.append(winner)
        bracket = next_round

    return bracket[0]

## 🛠️ Step 5: Monte Carlo Execution (10,000 Iterations)
Running the full simulation loop and aggregating results.

In [ ]:
from collections import Counter
import pandas as pd

# STEP 3: Execute the Monte Carlo Engine
N_SIMULATIONS = 10000
winners = []

print(f"Executing {N_SIMULATIONS} iterations on the uncompromised 48-team field...")
for i in range(N_SIMULATIONS):
    winner = run_tournament(final_teams)
    winners.append(winner)

# STEP 4: Export and Report
prob_df = pd.DataFrame(Counter(winners).items(), columns=['Team', 'Wins'])
prob_df['Probability'] = prob_df['Wins'] / N_SIMULATIONS
prob_df = prob_df.sort_values(by='Probability', ascending=False).reset_index(drop=True)

export_file = os.path.join(output_path, 'monte_carlo_results.csv')
prob_df.to_csv(export_file, index=False)

display(Markdown("### 🏆 Top 10 World Cup 2026 Winners (Flawless 48-Team Dataset)"))
display(prob_df.head(10))

Executing 10000 iterations on the uncompromised 48-team field...


### 🏆 Top 10 World Cup 2026 Winners (Flawless 48-Team Dataset)

,Team,Wins,Probability
0,Belgium,439,0.0439
1,Germany,402,0.0402
2,England,397,0.0397
3,Brazil,394,0.0394
4,France,376,0.0376
5,Algeria,374,0.0374
6,Portugal,372,0.0372
7,USA,369,0.0369
8,Japan,368,0.0368
9,Spain,367,0.0367
